# Extracting and Visualizing Stock Data
## Tesla (TSLA) and GameStop (GME)

**Author: Nikitha**  
**Coursera Peer-Graded Final Project**

This notebook retrieves historical stock data with yfinance, scrapes quarterly revenue tables with requests and BeautifulSoup, and builds stock-price-versus-revenue dashboards.

**Run in Google Colab:** Open the notebook, choose **Runtime → Run all**, wait for the cells to finish, then save the executed notebook. Capture screenshots showing both code and outputs for the Coursera submission.

**Important:** Financial websites can change their HTML or block automated requests. If a revenue-scraping cell reports an error, retry later or use the revenue data source provided in your course materials. Do not submit screenshots showing failed or empty results.

## 1. Install and import libraries

In [ ]:
%pip -q install yfinance beautifulsoup4 lxml html5lib plotly requests pandas

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup
from io import StringIO
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("Libraries imported successfully.")

## 2. Define the dashboard function

The function plots the stock closing price as a line and quarterly revenue as bars. Separate y-axes are used because the measurements have different units.

In [ ]:
def make_graph(stock_data, revenue_data, stock):
    stock_plot = stock_data.copy()
    revenue_plot = revenue_data.copy()

    stock_plot["Date"] = pd.to_datetime(stock_plot["Date"], errors="coerce")
    revenue_plot["Date"] = pd.to_datetime(revenue_plot["Date"], errors="coerce")
    stock_plot["Close"] = pd.to_numeric(stock_plot["Close"], errors="coerce")
    revenue_plot["Revenue"] = pd.to_numeric(revenue_plot["Revenue"], errors="coerce")

    stock_plot = stock_plot.dropna(subset=["Date", "Close"]).sort_values("Date")
    revenue_plot = revenue_plot.dropna(subset=["Date", "Revenue"]).sort_values("Date")

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Scatter(x=stock_plot["Date"], y=stock_plot["Close"],
                   name="Closing stock price", mode="lines"),
        secondary_y=False
    )
    fig.add_trace(
        go.Bar(x=revenue_plot["Date"], y=revenue_plot["Revenue"],
               name="Quarterly revenue"),
        secondary_y=True
    )
    fig.update_layout(
        title=f"{stock} Stock Price vs. Quarterly Revenue",
        xaxis_title="Date", hovermode="x unified",
        template="plotly_white", height=600
    )
    fig.update_yaxes(title_text="Stock price (USD)", secondary_y=False)
    fig.update_yaxes(title_text="Revenue (USD millions, where reported)", secondary_y=True)
    fig.show()

## 3. Question 1 – Extract Tesla stock data using yfinance

Create a Tesla ticker object, retrieve the maximum available historical data, reset the index, and display the first five rows.

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)

print("Tesla stock data – first five rows:")
display(tesla_data.head())

## 4. Question 2 – Scrape Tesla quarterly revenue

The scraper looks for a quarterly revenue table in the Macrotrends page and standardizes it into `Date` and `Revenue` columns. Revenue remains in the units displayed by the source (normally USD millions).

In [ ]:
def scrape_macrotrends_revenue(url, company_name):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/120.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    if not soup.find_all("table"):
        raise ValueError(
            f"No HTML tables found for {company_name}; the site may have changed "
            "its page structure or blocked automated access."
        )

    tables = pd.read_html(StringIO(response.text))
    candidates = []
    for table in tables:
        columns_text = " ".join(str(c).lower() for c in table.columns)
        sample_text = " ".join(table.astype(str).head(8).values.flatten()).lower()
        if "revenue" in columns_text or "revenue" in sample_text:
            candidates.append(table)

    if not candidates:
        raise ValueError(f"No revenue table could be identified for {company_name}.")

    raw = candidates[0].copy()
    raw.columns = [str(c).strip() for c in raw.columns]

    date_col = next(
        (c for c in raw.columns
         if any(word in c.lower() for word in ["quarter", "date", "period"])),
        raw.columns[0]
    )
    revenue_col = next((c for c in raw.columns if "revenue" in c.lower()), None)
    if revenue_col is None:
        if len(raw.columns) < 2:
            raise ValueError("The identified table does not contain a revenue column.")
        revenue_col = raw.columns[1]

    result = raw[[date_col, revenue_col]].copy()
    result.columns = ["Date", "Revenue"]
    result["Date"] = pd.to_datetime(result["Date"], errors="coerce")
    result["Revenue"] = (
        result["Revenue"].astype(str)
        .str.replace(r"[^0-9.\-]", "", regex=True)
    )
    result["Revenue"] = pd.to_numeric(result["Revenue"], errors="coerce")
    result = result.dropna(subset=["Date", "Revenue"])
    result = result.sort_values("Date").reset_index(drop=True)

    if result.empty:
        raise ValueError(f"No usable quarterly revenue rows were extracted for {company_name}.")
    return result

tesla_revenue_url = "https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue"

try:
    tesla_revenue = scrape_macrotrends_revenue(tesla_revenue_url, "Tesla")
    print("Tesla quarterly revenue – last five rows:")
    display(tesla_revenue.tail())
except Exception as error:
    print("Tesla revenue scraping did not complete:", error)
    print("Retry later or use the Tesla revenue source supplied in your course materials.")

## 5. Question 3 – Extract GameStop stock data using yfinance

Create a GameStop ticker object, retrieve the maximum available historical data, reset the index, and display the first five rows.

In [ ]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)

print("GameStop stock data – first five rows:")
display(gme_data.head())

## 6. Question 4 – Scrape GameStop quarterly revenue

Use the same scraper for the GameStop quarterly revenue page and display the last five rows.

In [ ]:
gme_revenue_url = "https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue"

try:
    gme_revenue = scrape_macrotrends_revenue(gme_revenue_url, "GameStop")
    print("GameStop quarterly revenue – last five rows:")
    display(gme_revenue.tail())
except Exception as error:
    print("GameStop revenue scraping did not complete:", error)
    print("Retry later or use the GameStop revenue source supplied in your course materials.")

## 7. Question 5 – Plot the Tesla stock and revenue dashboard

In [ ]:
if "tesla_revenue" in globals() and not tesla_revenue.empty:
    make_graph(tesla_data, tesla_revenue, "Tesla")
else:
    print("Tesla dashboard not plotted: first complete the Tesla revenue scraping cell.")

## 8. Question 6 – Plot the GameStop stock and revenue dashboard

In [ ]:
if "gme_revenue" in globals() and not gme_revenue.empty:
    make_graph(gme_data, gme_revenue, "GameStop")
else:
    print("GameStop dashboard not plotted: first complete the GameStop revenue scraping cell.")

## 9. Question 7 – Save and share the notebook using GitHub

1. In Google Colab, choose **File → Save a copy in Drive**.
2. Download the executed notebook using **File → Download → Download .ipynb**.
3. Create a GitHub repository (or use a repository you own).
4. Upload the `.ipynb` file to the repository.
5. Copy the repository or notebook URL and paste it into the Coursera GitHub URL field.

### Submission checklist
- [ ] Tesla stock data (`tesla_data.head()`)
- [ ] Tesla revenue data (`tesla_revenue.tail()`)
- [ ] GameStop stock data (`gme_data.head()`)
- [ ] GameStop revenue data (`gme_revenue.tail()`)
- [ ] Tesla dashboard
- [ ] GameStop dashboard
- [ ] Executed notebook uploaded to GitHub
- [ ] GitHub URL added to Coursera

### References
- yfinance documentation: https://ranaroussi.github.io/yfinance/
- Beautiful Soup documentation: https://www.crummy.com/software/BeautifulSoup/bs4/doc/
- Tesla revenue page: https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue
- GameStop revenue page: https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue

*Educational analysis only; not investment advice.*